# 18 - FilmAffinity TV ranking and mixed-feed policy

**Question:** Can a small change to the ranking policy improve the balance of movie and whole-series recommendations in one actual ten-title feed?

This is a bounded validation follow-up to [notebook 17](17_filmaffinity_hybrid_recommendations.ipynb). Reuse its frozen fitted basis and split, with a documented **metadata-only unit veto** applied consistently throughout notebook 18. Correct confirmed invalid whole-series units, then compare how movie and TV rankings are combined; do not retrain the model or modify notebook 17's saved artifacts.

There is no app update, new dataset, EASE search, LLM experiment, download, or paid API call in this study. The reserved FilmAffinity test feedback and all earlier MovieLens feedback stay closed.

## Prespecified boundary and data contract

- Verify the hashes of notebook 17's training and validation files and trusted fitted-model bundle before use. Keep the original source snapshots, split files, and saved model byte-unchanged. Reuse the fitted features and training-only prior rather than refitting them.
- Record the confirmed metadata-only unit-exclusion IDs and their source evidence in the study plan **before computing notebook 18 policy scores**. Apply this same veto to every candidate policy and corrected control: disable vetoed recommendation candidates, zero their profile-like columns, remove their invalid-unit ratings from validation targets, and recalculate training TV-like counts, history bins, and fixed task cohorts. Do not select exclusions from validation recommendation performance.
- A like remains **8/10 or higher**. Every training-rated title is excluded, including low ratings. Unobserved titles are not explicit dislikes.
- Read only the frozen training and validation artifacts needed for this follow-up. Do not reread the full raw ratings file or load reserved-test feedback.
- Keep the movie branch's model and weights fixed at notebook 17's saved best personalized hybrid, using profiles and candidate eligibility after the shared unit veto. Within TV, compare normalized popularity with that saved personalized ranking signal, with both normalized over **eligible unseen TV candidates**.
- TV blend weights are fixed in advance: **0, 0.25, 0.5, and 1**, plus **training TV-like count / (training TV-like count + 5)**. Weight 0 is popularity; weight 1 is personalized. The adaptive weight uses only training TV likes, never validation history or targets.
- Cross each of those five TV choices with **2, 4, or 6 TV positions in a total of ten recommendations**. Use even proportional interleaving and preserve each branch's original item ordering. Keep deterministic ties and the same treatment of unavailable branches.
- This makes **15 candidate policies**, plus the **Corrected popularity control** and **Corrected hybrid control**, for **17 comparisons**. These controls use notebook 17's saved model definitions under notebook 18's corrected catalog, profiles, labels, and cohorts; they are not the unchanged original notebook 17 rankings.
- First reproduce the original notebook 17 mixed-control metrics on its original artifacts as a separate regression check. Keep that check separate from policy selection and from the corrected-control leaderboard.

The hypothesis is that TV popularity may be more useful with limited TV history and personalization may help with more history. This is a hypothesis to measure, not an assumed result. Type allocation is a ranking policy, not a predicted probability that a user wants TV.

## Prespecified selection and diagnostics

Select a policy using the **balanced mean of movie Recall@10 and whole-series Recall@10 measured from the same actual mixed top-ten lists**. After the common metadata-only unit veto, fix each domain's eligible user cohort once and use it for every policy and both corrected controls; cohort membership must not depend on which recommendations a policy returns.

Remove only the documented invalid-unit validation ratings. Keep **all remaining relevant validation titles** in Recall and ideal-DCG denominators, including titles absent from the eligible candidate catalog. Report mixed Recall@10/NDCG@10, movie and TV metrics from the actual mixed list, and TV exposure. Break results down by **0, 1-4, and 5+ training TV likes**, recalculated after the veto, with cohort sizes and coverage alongside averages.

Separate movie-only and TV-only shelves are diagnostic comparisons. A ten-item shelf gives that domain ten slots; a mixed list allocates a shared ten-slot budget. Shelf results therefore cannot be presented as equivalent to the mixed-feed outcome.

Before interpreting unit labels, inspect a small manual source-unit spotcheck selected using training candidates only. A confirmed source-unit error triggers the documented shared veto; the remaining labels are still provisional. This is a focused check of source evidence, not a representative population error estimate or proof that all retained entries are whole series.

**Pre-score audit finding:** FilmAffinity item **745751, Stranger Things**, explicitly directs users to separately voted later seasons from season 2 onward. Treat this source entry as season 1 rather than a whole-series unit, and include it in the confirmed unit exclusions. Inspect the saved evidence and any additional confirmed exclusions below.

In [ ]:
from pathlib import Path
import sys

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "scripts" / "filmaffinity_experiment.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the movie-recommendation-system project.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.filmaffinity_ranking_policy import run_experiment

report = run_experiment(ROOT)

## Frozen fitted basis, unit veto, and study plan

The following records come from this run rather than from handwritten results. Review the original input identities, the pre-score unit-veto list, the selection rule, the corrected fixed task cohorts, and the policy definitions before reading the leaderboards.

In [ ]:
from IPython.display import display
import pandas as pd

OUTPUT = ROOT / "data" / "processed" / "notebook18_filmaffinity"

display(report.get("scope", {"status": "No scope section was supplied by the helper."}))
display(report.get("study_plan", {"status": "No study-plan section was supplied by the helper."}))
display(report.get("dataset", {"status": "No dataset section was supplied by the helper."}))

## Provisional source-unit spotcheck and coverage

FilmAffinity item units were inferred in the earlier source audit. Series, seasons, episodes, miniseries, and films can have inconsistent or ambiguous source descriptions. The training-selected source-unit check identified confirmed errors, including the Stranger Things season-1 entry, which this follow-up explicitly vetoes. The saved exclusion table records IDs and source evidence. The small sample cannot establish the accuracy of every remaining TV label.

Coverage describes which valid-unit validation targets the training catalog can retrieve after the shared veto. Confirmed invalid-unit ratings are removed from the targets consistently for every policy and control; all other unavailable targets still contribute to the evaluation denominator. Fitted vocabularies and feature weights still come from notebook 17's original fit, which included the excluded items' metadata. This is a documented eligibility and profile correction, not a clean refit of the model.

In [ ]:
def show_saved_table(filename):
    path = OUTPUT / filename
    if path.is_file():
        frame = pd.read_csv(path)
        display(frame)
        return frame
    print(f"Artifact not available: {path.name}")
    return None

display(report.get("unit_spotcheck", {"status": "No spotcheck summary was supplied by the helper."}))
unit_spotcheck_table = show_saved_table("unit_spotcheck.csv")
display(report.get("unit_corrections", {"status": "No unit-correction summary was supplied by the helper."}))
confirmed_unit_exclusions_table = show_saved_table("confirmed_unit_exclusions.csv")
display(report.get("coverage", {"status": "No coverage section was supplied by the helper."}))

## Actual mixed top-ten policy comparison

Every row below must use the same corrected candidate eligibility, profiles, validation labels, and fixed movie and TV cohorts. Use the balanced domain-recall objective to select a policy, then inspect mixed-list accuracy and exposure together. Greater TV exposure alone does not demonstrate more relevant recommendations.

The Corrected popularity control and Corrected hybrid control provide a reference for introducing explicit type allocation after the common metadata-only unit veto. The original notebook 17 control reproduction is a separate regression check. Corrected results are not directly comparable to the old raw scores because item eligibility, target labels, profiles, and cohorts have changed. An apparent gain on this validation set is exploratory evidence for the policy and objective tested here.

In [ ]:
policy_leaderboard_table = show_saved_table("policy_leaderboard.csv")
display(report.get("best_policy", {"status": "No selected-policy section was supplied by the helper."}))
display(report.get("controls", {"status": "No control summaries were supplied by the helper."}))

## Accuracy and TV-exposure tradeoff

Interpret this plot using the actual mixed-list budget of ten titles. Inspect the labeled controls and selected policy rather than treating TV slot allocation as a calibrated preference probability.

In [ ]:
from IPython.display import Image

tradeoff_path = OUTPUT / "mixed_policy_tradeoff.png"
if tradeoff_path.is_file():
    display(Image(filename=str(tradeoff_path)))
else:
    print("The mixed-policy tradeoff plot was not produced by this run.")

## Results by training TV-history size

The **0, 1-4, and 5+** bins use only the user's valid-unit training TV likes, recalculated after removing vetoed profile likes. Validation likes determine corrected evaluation targets, not the adaptive policy's history count. Read the cohort sizes and available-target coverage with each bin's metrics.

A small or empty subgroup cannot support a broad conclusion about cold start or cross-domain transfer. The adaptive weight is useful only if its measured tradeoff supports the hypothesis.

In [ ]:
history_bin_table = show_saved_table("history_bin_metrics.csv")
display(report.get("history_bins", {"status": "No history-bin summary was supplied by the helper."}))

## Separate shelves: a different slot budget

The following movie-only and TV-only shelf diagnostics each devote their own recommendation budget to one domain. They help identify whether candidate ranking or mixed-feed allocation limits retrieval, but they do not share the mixed feed's ten total positions.

Do not compare a TV-only Recall@10 directly with TV recall from a mixed list as though the TV budgets were equal.

In [ ]:
shelf_leaderboard_table = show_saved_table("shelf_leaderboard.csv")
display(report.get("best_shelf_model", {"status": "No shelf-model summary was supplied by the helper."}))

## Interpretation and limits

This follow-up selects a **ranking policy under a new mixed-feed objective**. Notebook 17's earlier primary objective used domain-specific ranking tasks; changing the objective changes what is optimized. Results here do not retroactively establish that notebook 17 selected the best mixed-feed policy.

The split follows review-publication date groups. These dates are proxies for preference-recording time, not verified rating timestamps. Catalog metadata is a current snapshot, and the earlier heuristic whole-TV-unit labels retain their limitations. The small source-unit spotcheck does not remove those limitations.

The original notebook 17 snapshots, split, and model remain byte-unchanged. Notebook 18 reuses that fitted basis with a documented metadata-only unit veto, so its valid candidate set, profile likes, validation labels, history bins, and cohorts differ from the original experiment. The fitted vocabularies still include metadata from excluded items. Differences among notebook 18 rows concern its common corrected population and ranking policies, not a newly trained model; differences from the old raw notebook 17 scores cannot be attributed solely to ranking policy. No conclusion about beating an LLM, the earlier MovieLens models, or live app recommendations follows from these results.

After bounded validation selection, a final untouched holdout or prospective user evaluation is needed to assess generalization. Keep the reserved test closed throughout this notebook. Do not repeatedly retune policies against it.

In [ ]:
display(report.get("files", {"status": "No output-file manifest was supplied by the helper."}))
print("Output directory:", OUTPUT)